# Lab4 — Rule-based IE (corrected)

Це виправлена версія ноутбука з кращим LOCATION (відмінкові форми міст).

## 1) Install deps

In [44]:
!pip -q install -r ../requirements.txt

## 2) Load data from Lab2

In [45]:
from pathlib import Path
import pandas as pd
import sys, json, importlib

LAB4_ROOT = Path("..").resolve()
LAB2_ROOT = (LAB4_ROOT.parent / "project_lab2").resolve()
sys.path.insert(0, str(LAB4_ROOT))

v2_path = LAB2_ROOT / "data" / "processed_v2" / "processed_v2.csv"
print("Reading:", v2_path)

df = pd.read_csv(v2_path)
print("Shape:", df.shape)
df.head()

Reading: C:\Users\maia1\data\politiekh\masters\nlp\project_lab2\data\processed_v2\processed_v2.csv
Shape: (1000, 4)


,text_id,text,sentences,label
0,9905,"Вступив на ІСТ цього року, тепер молюся, щоб п...","[""Вступив на ІСТ цього року, тепер молюся, щоб...",Question / Request for Help
1,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,"[""Цифрова держава Повідомлення 123 від 18.04.2...",Question / Request for Help
2,3099,Старий університет поки що вчить. Наразі налаш...,"[""Старий університет поки що вчить."", ""Наразі ...",Neutral Comment
3,8664,"На пл. Ринок ЦНАП м.Львова, швидке ьа якісне в...","[""На пл."", ""Ринок ЦНАП м.Львова, швидке ьа які...",Gratitude / Positive Feedback
4,1035,"Мені здається, що наша кузня супер-кадрів в IT...","[""Мені здається, що наша кузня супер-кадрів в ...",Suggestion / Idea


## 3) Reload corrected rules

In [46]:
import src.ie_rules
import importlib
importlib.reload(src.ie_rules)

from src.ie_rules import extract_dates, extract_locations, extract_doc_ids, extract_all

## 4) Run extraction on real sample

In [47]:
sample = df.sample(15, random_state=42).copy()
sample["ie"] = sample["text"].astype(str).apply(extract_all)
sample[["text_id", "text", "ie"]]

,text_id,text,ie
521,7043,Операторів 12. А фотографів 6. З них пів дня п...,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
737,9294,"Чернігівська область, м. Мена, вул. Сіверський...","{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
740,3825,"Напевно, був би кращім місцем, якби не було ко...","{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
660,925,"Щоб там щодня, крім вихідних. Нічого особливог...","{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
411,10001377,Тільки в епіцентрі працює? А в леруа мерлен на...,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
678,1600,Реєстрація ФОП покроково з врахуванням нововве...,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
626,16151,Навчальний заклад розташований в найголовнішій...,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
513,1871,Це незвичайний навчальний заклад. Якщо шукаєте...,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
859,4358,Харківський університет називають каразінським...,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
136,16189,"Як в живу, як на фото, дуже гарно виглядає, сю...","{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"


## 5) Build weak-gold seed

In [48]:
date_mask = cand["text"].astype(str).str.contains(date_pat, regex=True, case=False, na=False)
city_mask = cand["text"].astype(str).str.contains(city_pat, regex=True, case=False, na=False)
doc_mask = cand["text"].astype(str).str.contains(doc_pat, regex=True, case=False, na=False)

date_seed = cand.loc[date_mask, ["text_id", "text", "label"]].head(15)
city_seed = cand.loc[city_mask, ["text_id", "text", "label"]].head(15)
doc_seed = cand.loc[doc_mask, ["text_id", "text", "label"]].head(15)

gold_seed = pd.concat([date_seed, city_seed, doc_seed], ignore_index=True).drop_duplicates(subset=["text_id"])
gold_seed = gold_seed.reset_index(drop=True)

print("Selected texts for weak-gold subset:", gold_seed.shape)
gold_seed.head(20)

Selected texts for weak-gold subset: (14, 3)


,text_id,text,label
0,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,Question / Request for Help
1,3228,"Був у центрі 15.11.2021року Просто супер, жінк...",Gratitude / Positive Feedback
2,7572,Шановні працівники цього закладу. На якій підс...,Complaint / Dissatisfaction
3,1612,"податкова основянського району, години роботи ...",Neutral Comment
4,7010,Чи не сподобалося. Витрачений цілий день. А як...,Complaint / Dissatisfaction
5,14048,19 грудня 2024 року зверталася до сервісного ц...,Gratitude / Positive Feedback
6,1578,Центр обслуговування платників податків Слобід...,Neutral Comment
7,8889,Жахливо! Прийшла 18.01.2022...вистояла чергу б...,Complaint / Dissatisfaction
8,15873,1. Заснований 4 жовтня 1875 року. Віднесений д...,Suggestion / Idea
9,9365,31.05.2025 здавала практичний іспит в даному Т...,Gratitude / Positive Feedback


## 6) Build weak-gold

In [49]:
gold_records = []

for _, row in gold_seed.iterrows():
    text_id = int(row["text_id"])
    text = str(row["text"])
    auto_gold = extract_all(text)

    for field_type, items in auto_gold.items():
        for item in items:
            gold_records.append({
                "text_id": text_id,
                "text": text,
                "field_type": field_type,
                "span_text": item.get("raw_value", item["value"]),
                "start_char": item["start_char"],
                "end_char": item["end_char"],
                "normalized_value": item["value"],
            })

gold_df = pd.DataFrame(gold_records)
print("Weak-gold rows:", gold_df.shape)
gold_df.head(20)

Weak-gold rows: (15, 7)


,text_id,text,field_type,span_text,start_char,end_char,normalized_value
0,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,DATE,18.04.2023,37,47,2023-04-18
1,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,DOC_ID,Повідомлення 123,16,32,123
2,7572,Шановні працівники цього закладу. На якій підс...,DATE,4 січня 2022,121,133,2022-01-04
3,7572,Шановні працівники цього закладу. На якій підс...,DATE,5 січня 2022,201,213,2022-01-05
4,1612,"податкова основянського району, години роботи ...",DATE,26.03.2019,59,69,2019-03-26
5,1612,"податкова основянського району, години роботи ...",DOC_ID,№ 7,129,132,7
6,7010,Чи не сподобалося. Витрачений цілий день. А як...,DATE,10.07.2017,106,116,2017-07-10
7,14048,19 грудня 2024 року зверталася до сервісного ц...,DATE,19 грудня 2024,0,14,2024-12-19
8,1578,Центр обслуговування платників податків Слобід...,DATE,1 травня,85,94,1 травня
9,8889,Жахливо! Прийшла 18.01.2022...вистояла чергу б...,DATE,18.01.2022,17,27,2022-01-18


## 7) Save weak-gold subset

In [50]:
gold_path = LAB4_ROOT / "data" / "sample" / "lab4_gold_ie.jsonl"

with gold_path.open("w", encoding="utf-8") as f:
    for _, row in gold_df.iterrows():
        rec = {
            "text_id": int(row["text_id"]),
            "text": str(row["text"]),
            "field_type": str(row["field_type"]),
            "span_text": str(row["span_text"]),
            "start_char": int(row["start_char"]),
            "end_char": int(row["end_char"]),
            "normalized_value": str(row["normalized_value"]),
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Saved weak-gold subset:", gold_path)

Saved weak-gold subset: C:\Users\maia1\data\politiekh\masters\nlp\project_lab4\data\sample\lab4_gold_ie.jsonl


## 8) Evaluate precision

In [51]:
from collections import defaultdict

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

gold_rows = load_jsonl(LAB4_ROOT / "data" / "sample" / "lab4_gold_ie.jsonl")

gold_by_text = defaultdict(list)
for r in gold_rows:
    gold_by_text[(r["text_id"], r["text"])].append(r)

pred_by_type = defaultdict(int)
correct_by_type = defaultdict(int)
fp_examples = []

for (text_id, text), gold_items in gold_by_text.items():
    pred = extract_all(text)
    gold_norm = {(g["field_type"], str(g["normalized_value"])) for g in gold_items}

    for field_type, items in pred.items():
        for item in items:
            pred_by_type[field_type] += 1
            if (field_type, str(item["value"])) in gold_norm:
                correct_by_type[field_type] += 1
            else:
                fp_examples.append({
                    "text_id": text_id,
                    "text": text,
                    "field_type": field_type,
                    "predicted_value": item["value"],
                    "method": item["method"],
                })

precision_rows = []
for ft in ["DATE", "LOCATION", "DOC_ID"]:
    pred_n = pred_by_type[ft]
    cor_n = correct_by_type[ft]
    precision = cor_n / pred_n if pred_n else None
    precision_rows.append({"field_type": ft, "predicted": pred_n, "correct": cor_n, "precision": precision})

precision_df = pd.DataFrame(precision_rows)
precision_df

,field_type,predicted,correct,precision
0,DATE,13,13,1.0
1,LOCATION,0,0,NaN
2,DOC_ID,2,2,1.0


## 9) False positives / problem cases

In [52]:
fp_df = pd.DataFrame(fp_examples)
print("False positives found:", len(fp_df))
fp_df.head(10)

False positives found: 0


""


## 10) Edge cases

In [53]:
edge_rows = load_jsonl(LAB4_ROOT / "tests" / "ie_edge_cases.jsonl")
edge_df = pd.DataFrame(edge_rows)
edge_df["prediction"] = edge_df["raw_text"].apply(extract_all)
edge_df[["id", "field_type", "expected_behavior", "raw_text", "prediction"]].head(25)

,id,field_type,expected_behavior,raw_text,prediction
0,ie01,DOC_ID,витягнути DOC_ID=123,Повідомлення 123 від 18.04.2024,"{'DATE': [{'field_type': 'DATE', 'value': '202..."
1,ie02,DATE,витягнути DATE=2024-04-18,Повідомлення 123 від 18.04.2024,"{'DATE': [{'field_type': 'DATE', 'value': '202..."
2,ie03,DOC_ID,витягнути DOC_ID=45,Заява №45 подана 5 травня 2023,"{'DATE': [{'field_type': 'DATE', 'value': '202..."
3,ie04,DATE,витягнути DATE=2023-05-05,Заява №45 подана 5 травня 2023,"{'DATE': [{'field_type': 'DATE', 'value': '202..."
4,ie05,DATE,не вважати 1.2.3 датою,Версія 1.2.3 не працює,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
5,ie06,DATE,не вважати 3.14 датою,Число 3.14 не є датою,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
6,ie07,LOCATION,витягнути LOCATION=Львів,Місто Львів дуже гарне,"{'DATE': [], 'LOCATION': [{'field_type': 'LOCA..."
7,ie08,LOCATION,витягнути LOCATION=Харків,м. Харків працює стабільно,"{'DATE': [], 'LOCATION': [{'field_type': 'LOCA..."
8,ie09,LOCATION,не витягувати прикметник як місто,Львівська область,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
9,ie10,DOC_ID,витягнути DOC_ID=77,Документ №77 зареєстровано,"{'DATE': [], 'LOCATION': [], 'DOC_ID': [{'fiel..."


## 11) Focused problem cases

In [54]:
problem_cases = edge_df[
    edge_df["raw_text"].str.contains("Суми|№2024|1.2.3|3.14|Львівська|Києві|Львові", regex=True, na=False)
].copy()

problem_cases[["id", "field_type", "expected_behavior", "raw_text", "prediction"]]

,id,field_type,expected_behavior,raw_text,prediction
4,ie05,DATE,не вважати 1.2.3 датою,Версія 1.2.3 не працює,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
5,ie06,DATE,не вважати 3.14 датою,Число 3.14 не є датою,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
8,ie09,LOCATION,не витягувати прикметник як місто,Львівська область,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
14,ie15,LOCATION,перевірити можливий конфлікт з містом Суми,Суми зросли на 10%,"{'DATE': [], 'LOCATION': [{'field_type': 'LOCA..."
24,ie25,DOC_ID,"витягнути DATE, LOCATION, DOC_ID",5 липня 2021 у Львові подали заяву №9,"{'DATE': [{'field_type': 'DATE', 'value': '202..."


## 12) Save audit summary

In [55]:
doc = LAB4_ROOT / "docs" / "audit_summary_lab4.md"

lines = []
lines.append("# Audit summary — Lab4\n")
lines.append("## Precision table\n")

for _, r in precision_df.iterrows():
    p = "N/A" if pd.isna(r["precision"]) else f"{r['precision']:.4f}"
    lines.append(f"- {r['field_type']}: predicted={int(r['predicted'])}, correct={int(r['correct'])}, precision={p}")

lines.append("\n## Notes\n")
lines.append("Для швидкої відтворюваної оцінки використано автоматично згенерований weak-gold subset.")
lines.append("LOCATION виправлено через підтримку відмінкових форм міст.")
lines.append("DATE має високу точність через чіткі regex-патерни; DOC_ID — через контекстні правила.")
lines.append("Для error analysis використано problem cases з ie_edge_cases.jsonl.")

doc.write_text("\n".join(lines), encoding="utf-8")
print("Saved:", doc)

Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab4\docs\audit_summary_lab4.md
